# 🚀 The ULTIMATE Whisper Small Fine-Tuning Pipeline

This version is designed specifically to handle **7-10+ hours** of dataset generation without crashing Google Colab.

### Key Upgrades in this version:
- **Automatic Disk Cleanup:** Audio files (`.wav`) are massive. This script deletes them immediately after their spectrogram features are generated to prevent Colab from running out of disk space!
- **Google Drive Integration (Optional):** Automatically saves the finished fine-tuned model to your Google Drive to prevent data loss if the session disconnects.
- **Bulletproof Transcripts:** Automatically parses all languages, converts `nl`, `es`, `de`, `fr`, etc. native subtitles to `en`, and safely formats them into JSON dictionaries.
- **Batch Safety:** If a corrupted chunk fails during processing, it intelligently logs it and continues processing the rest of the 500+ chunks instead of crashing the notebook.

**Important:** Change runtime type > Hardware accelerator > **T4 GPU**.

In [ ]:
# @title Step 0: Mount Google Drive (Recommended but Optional)
# This ensures your 1-2 hour trained model is safely stored on your Drive instantly.

from google.colab import drive
try:
    drive.mount('/content/drive')
    print("\n✅ Google Drive mounted correctly! Models will be saved here.")
    DRIVE_AVAILABLE = True
except Exception as e:
    print("\n⚠️ Could not mount Drive. Models will be saved to local Colab storage only.")
    DRIVE_AVAILABLE = False

Mounted at /content/drive

✅ Google Drive mounted correctly! Models will be saved here.


## Step 1: Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio
!pip install transformers datasets accelerate evaluate jiwer librosa soundfile yt-dlp youtube-transcript-api tqdm numpy pydub
!apt-get install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 122.4 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


## Step 2: Download Scripts
We combine everything into two highly optimized scripts.

In [ ]:
%%writefile 1_download_and_extract.py
import os
import json
import logging
from typing import List, Dict, Optional
import yt_dlp
from youtube_transcript_api import YouTubeTranscriptApi

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

DATA_DIR = os.path.join(os.getcwd(), 'dataset', 'raw')
AUDIO_DIR = os.path.join(DATA_DIR, 'audio')
TRANSCRIPT_DIR = os.path.join(DATA_DIR, 'transcripts')
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(TRANSCRIPT_DIR, exist_ok=True)

def extract_video_id(url: str) -> str:
    if "youtu.be" in url: return url.split("/")[-1].split("?")[0]
    if "v=" in url: return url.split("v=")[1].split("&")[0]
    return url

def get_urls_from_input(url: str) -> List[str]:
    ydl_opts = {'extract_flat': 'in_playlist', 'quiet': True, 'no_warnings': True}
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=False)
            if 'entries' in info:
                logger.info(f"Playlist detected! Found {len(info['entries'])} videos.")
                return [entry['url'] for entry in info['entries'] if entry.get('url')]
            return [url]
    except Exception as e:
        logger.error(f"Failed to extract info from {url}. Ensure it is a valid link (e.g. ?list=ID). Error: {e}")
        return []

def download_youtube_audio(youtube_url: str, output_path: str) -> bool:
    logger.info(f"Downloading audio from {youtube_url}...")
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'wav'}],
        'postprocessor_args': ['-ar', '16000', '-ac', '1'], # 16kHz mono required for Whisper
        'outtmpl': output_path, # Fixed output template (.wav gets added by yt-dlp automatically)
        'quiet': False,
        'no_warnings': True,
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl: ydl.download([youtube_url])
        return True
    except Exception as e:
        logger.error(f"Failed to download audio for {youtube_url}: {e}")
        return False

def download_youtube_transcript(video_id: str, output_path: str, languages: List[str] = ['en', 'nl', 'es', 'de', 'fr', 'ja', 'it', 'ko', 'zh-Hans', 'ru']) -> Optional[List[Dict]]:
    logger.info(f"Fetching transcript for video ID: {video_id}...")
    try:
        # Fixed: Usage of instance method based on modern youtube_transcript_api versions!!!
        transcript_list = YouTubeTranscriptApi().list(video_id)

        # Try to find transcript in our widespread supported language list
        transcript = transcript_list.find_transcript(languages)

        # Always normalize and translate to English to guarantee Whisper learns English text mapping
        if transcript.language_code != 'en':
            logger.info(f"Translating video {video_id} ({transcript.language_code}) native transcript to English...")
            transcript = transcript.translate('en')

        fetched_data = transcript.fetch()

        # Fixed: Safely enforce JSON Serialization from FetchedTranscriptSnippet using attribute dot-notation
        data = [
            {"text": t.text, "start": t.start, "duration": t.duration}
            for t in fetched_data
        ]

        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        return data

    except Exception as e:
        logger.error(f"Failed to fetch transcript for {video_id}: {e}")
        return None

def process_video(url: str):
    video_id = extract_video_id(url)
    logger.info(f"\n{'='*40}\nProcessing Video ID: {video_id}\n{'='*40}")

    transcript_data = download_youtube_transcript(video_id, os.path.join(TRANSCRIPT_DIR, f"{video_id}.json"))
    if not transcript_data:
        logger.warning(f"Skipping {video_id} due to missing transcripts.")
        return False

    audio_path = os.path.join(AUDIO_DIR, video_id)
    if not os.path.exists(f"{audio_path}.wav"):
        if not download_youtube_audio(url, audio_path): return False
    return True

if __name__ == "__main__":
    import sys
    input_urls = [u.strip() for u in sys.argv[1].split(",") if u.strip()]
    target_urls = []
    for u in input_urls: target_urls.extend(get_urls_from_input(u))
    logger.info(f"Total discovered videos to process: {len(target_urls)}")
    for url in target_urls: process_video(url)
    logger.info("\n\n✅ Phase 1: Downloading & Extraction Completed! ✓")

Writing 1_download_and_extract.py


In [ ]:
%%writefile 2_build_dataset.py
import os
import json
import logging
from pydub import AudioSegment
from datasets import Dataset, DatasetDict
from transformers import WhisperFeatureExtractor, WhisperTokenizer
import warnings
import shutil

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'dataset', 'raw')
PROCESSED_DATA_DIR = os.path.join(BASE_DIR, 'dataset', 'processed')
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

MODEL_ID = "openai/whisper-small"
MAX_DURATION = 30000  # 30 seconds explicitly
MIN_DURATION = 500    # 500 ms minimum

def main():
    logger.info("🚀 Initializing High-Volume Preprocessing Pipeline...")
    feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
    tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID, language="english", task="transcribe")

    # Step A: Identify valid transcripts & audio
    valid_pairs = []
    if not os.path.exists(os.path.join(DATA_DIR, 'transcripts')):
        logger.error("Dataset raw directories do not exist! Did Step 1 finish successfully?")
        return

    for f in os.listdir(os.path.join(DATA_DIR, 'transcripts')):
        if f.endswith('.json'):
            vid = f.replace('.json', '')
            audio_path = os.path.join(DATA_DIR, 'audio', f"{vid}.wav")
            if os.path.exists(audio_path):
                valid_pairs.append((vid, audio_path, os.path.join(DATA_DIR, 'transcripts', f)))

    if not valid_pairs:
        logger.error("❌ 0 valid Audio + Transcript pairs detected!")
        return

    logger.info(f"🔪 Slicing {len(valid_pairs)} master videos into Whisper 30s Chunks...")
    dataset_rows = []

    # Step B: Slice Audio
    for vid, audio_path, trans_path in valid_pairs:
        with open(trans_path, 'r') as file:
            transcript_data = json.load(file)

        try:
            audio = AudioSegment.from_wav(audio_path)
        except Exception as e:
            logger.warning(f"Could not load Audio file for {vid}: {e}. Skipping.")
            continue

        chunk_dir = os.path.join(PROCESSED_DATA_DIR, 'chunks', vid)
        os.makedirs(chunk_dir, exist_ok=True)

        successful_chunks = 0
        for i, seg in enumerate(transcript_data):
            # Safely check missing array properties for malformed YouTube captions
            if 'start' not in seg or 'duration' not in seg or 'text' not in seg: continue
            if seg['start'] is None or seg['duration'] is None: continue

            start = int(seg['start'] * 1000)
            dur = int(seg['duration'] * 1000)
            text = str(seg['text']).strip()

            # Restrict duration sizing
            if not text or dur < MIN_DURATION or dur > MAX_DURATION: continue

            chunk_path = os.path.join(chunk_dir, f"{vid}_chunk_{i}.wav")
            try:
                # Explicitly conform to Whisper format and save instantly
                sliced_audio = audio[start:start+dur]
                sliced_audio.set_frame_rate(16000).set_channels(1).export(chunk_path, format="wav")
                dataset_rows.append({"audio_path": chunk_path, "text": text})
                successful_chunks += 1
            except Exception as e:
                pass # Suppress minor slicing corruptions, keep going

        logger.info(f" > Video {vid}: Generated {successful_chunks} clean training chunks.")

        # 🧹 AGGRESSIVE DISK CLEANUP: Delete the massive raw 1-2GB origin WAV file after slicing immediately to retain disk space!
        os.remove(audio_path)

    if not dataset_rows:
        logger.error("❌ 0 total audio chunks properly identified. Check your video URLs.")
        return

    logger.info(f"\n🔥 Ready to process {len(dataset_rows)} Total Training Chunks!")
    raw_dataset = Dataset.from_list(dataset_rows)

    def load_audio(batch):
        import librosa
        speech_array, sr = librosa.load(batch["audio_path"], sr=16000)
        batch["audio"] = {"array": speech_array, "sampling_rate": sr}
        return batch

    def prepare_dataset(b):
        b["input_features"] = feature_extractor(b["audio"]["array"], sampling_rate=16000).input_features[0]
        b["labels"] = tokenizer(b["text"]).input_ids
        return b

    # Step C: LogMel Spectrogram Extraction
    logger.info("Extracting audio matrices... (This is CPU bound and may take 20-30+ minutes for huge datasets)")
    # Removed num_proc=2 to prevent Colab Memory Crash (BrokenPipeError) when loading 5800+ audio files into RAM.
    audio_dataset = raw_dataset.map(load_audio)

    logger.info("Tokenizing words to integer matrices...")
    processed_dataset = audio_dataset.map(prepare_dataset, remove_columns=audio_dataset.column_names)

    # 🧹 AGGRESSIVE DISK CLEANUP: Delete the small chunk files entirely since features are now in memory.
    logger.info("Wiping intermediate disk files to free storage space...")
    try:
        shutil.rmtree(os.path.join(PROCESSED_DATA_DIR, 'chunks'))
    except: pass

    # Step D: Final Output
    split_dataset = processed_dataset.train_test_split(test_size=0.1)
    save_path = os.path.join(PROCESSED_DATA_DIR, "whisper_dataset")
    split_dataset.save_to_disk(save_path)

    logger.info(f"\n✅ Phase 2: Processing Complete! Dataset safely housed mathematically at: {save_path}")

if __name__ == "__main__":
    main()

Writing 2_build_dataset.py


## Step 3: Configure Target Videos & Run Downloads
Specify your PLAYLIST link or individual videos separated by commas. Wait for it to finish downloading the MP3s and formatting the English Transcripts.

In [ ]:
YOUTUBE_URLS = "https://www.youtube.com/playlist?list=PLsRNoUx8w3rPPvAvyw4Uj_XTU-1WqD8Et"

!python 1_download_and_extract.py "{YOUTUBE_URLS}"

2026-03-09 16:02:28,306 - __main__ - INFO - Playlist detected! Found 25 videos.
2026-03-09 16:02:28,308 - __main__ - INFO - Total discovered videos to process: 25
2026-03-09 16:02:28,309 - __main__ - INFO - 
Processing Video ID: eVFzbxmKNUw
2026-03-09 16:02:28,309 - __main__ - INFO - Fetching transcript for video ID: eVFzbxmKNUw...
2026-03-09 16:02:29,270 - __main__ - INFO - Downloading audio from https://www.youtube.com/watch?v=eVFzbxmKNUw...
[youtube] Extracting URL: https://www.youtube.com/watch?v=eVFzbxmKNUw
[youtube] eVFzbxmKNUw: Downloading webpage
[youtube] eVFzbxmKNUw: Downloading android vr player API JSON
[info] eVFzbxmKNUw: Downloading 1 format(s): 251
[download] Destination: /content/dataset/raw/audio/eVFzbxmKNUw
[download] 100% of   11.46MiB in 00:00:00 at 47.36MiB/s
[ExtractAudio] Destination: /content/dataset/raw/audio/eVFzbxmKNUw.wav
Deleting original file /content/dataset/raw/audio/eVFzbxmKNUw (pass -k to keep)
2026-03-09 16:02:41,821 - __main__ - INFO - 
Processing Vi

## Step 4: Build The Hugging Face Dataset
Converts audio into math! Slices everything up and maps tokens. (Will take several minutes)

In [ ]:
!python 2_build_dataset.py

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
2026-03-09 16:05:17,929 - 🚀 Initializing High-Volume Preprocessing Pipeline...
2026-03-09 16:05:18,268 - HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-03-09 16:05:18,503 - HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/preprocessor_config.j

In [ ]:
!pip install -U transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 21.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Step 5: GPU Model Fine-Tuning
Configure your batch size and steps. We use FP16 Mixed Precision for the fastest training speeds to fully utilize the T4 Tensor Cores.

In [ ]:
%%writefile 3_train_whisper.py

import os
import torch
import logging
from dataclasses import dataclass
from typing import Any

from datasets import load_from_disk
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

import evaluate

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# -------------------------------------------------
# Paths
# -------------------------------------------------

BASE_DIR = os.getcwd()

PROCESSED_DATA_DIR = os.path.join(
    BASE_DIR,
    "dataset",
    "processed",
    "whisper_dataset"
)

DRIVE_PATH = "/content/drive/MyDrive/MeetBuddy_Whisper_Models"

if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(DRIVE_PATH, exist_ok=True)
    OUTPUT_DIR = os.path.join(
        DRIVE_PATH,
        "meetbuddy-whisper-small-finetuned"
    )
else:
    OUTPUT_DIR = os.path.join(
        BASE_DIR,
        "meetbuddy-whisper-small-finetuned"
    )

MODEL_ID = "openai/whisper-small"


# -------------------------------------------------
# Data Collator
# -------------------------------------------------

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:

    processor: Any

    def __call__(self, features):

        input_features = [
            {"input_features": f["input_features"]}
            for f in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [
            {"input_ids": f["labels"]}
            for f in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        if (
            labels[:, 0]
            == self.processor.tokenizer.bos_token_id
        ).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


# -------------------------------------------------
# Training
# -------------------------------------------------

def main():

    logger.info(
        f"Loading dataset from {PROCESSED_DATA_DIR}"
    )

    dataset = load_from_disk(PROCESSED_DATA_DIR)

    processor = WhisperProcessor.from_pretrained(
        MODEL_ID,
        language="English",
        task="transcribe"
    )

    model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_ID
    )

    model.config.forced_decoder_ids = None
    model.config.suppress_tokens = []

    data_collator = DataCollatorSpeechSeq2SeqWithPadding(
        processor=processor
    )

    metric = evaluate.load("wer")


    # -------------------------------------------------
    # Metrics
    # -------------------------------------------------

    def compute_metrics(pred):

        pred_ids = pred.predictions
        label_ids = pred.label_ids

        label_ids[label_ids == -100] = (
            processor.tokenizer.pad_token_id
        )

        pred_str = processor.tokenizer.batch_decode(
            pred_ids,
            skip_special_tokens=True
        )

        label_str = processor.tokentrash:///Ultimate_Whisper_Fine_Tuning_Colab.ipynbizer.batch_decode(
            label_ids,
            skip_special_tokens=True
        )

        wer = 100 * metric.compute(
            predictions=pred_str,
            references=label_str
        )

        return {"wer": wer}


    # -------------------------------------------------
    # Training Arguments
    # -------------------------------------------------

    training_args = Seq2SeqTrainingArguments(

        output_dir=OUTPUT_DIR,

        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,

        gradient_accumulation_steps=2,

        learning_rate=1e-5,

        warmup_steps=100,

        max_steps=3000,

        gradient_checkpointing=True,

        fp16=True,

        eval_strategy="steps",

        predict_with_generate=True,

        generation_max_length=225,

        save_steps=200,

        eval_steps=200,

        logging_steps=50,

        load_best_model_at_end=True,

        metric_for_best_model="wer",

        greater_is_better=False,

        save_total_limit=2,

        remove_unused_columns=False
    )


    # -------------------------------------------------
    # Trainer
    # -------------------------------------------------

    trainer = Seq2SeqTrainer(

        args=training_args,

        model=model,

        train_dataset=dataset["train"],

        eval_dataset=dataset["test"],

        data_collator=data_collator,

        compute_metrics=compute_metrics,

        processing_class=processor
    )


    torch.set_float32_matmul_precision("high")

    logger.info(
        "\n🔥 STARTING WHISPER TRAINING 🔥"
    )

    trainer.train()


    # -------------------------------------------------
    # Save Model
    # -------------------------------------------------

    model.save_pretrained(OUTPUT_DIR)

    processor.save_pretrained(OUTPUT_DIR)

    logger.info(
        f"\n✅ Model saved to {OUTPUT_DIR}"
    )


# -------------------------------------------------

if __name__ == "__main__":

    main()

Overwriting 3_train_whisper.py


In [ ]:
!python 3_train_whisper.py

INFO:__main__:Loading dataset from /content/dataset/processed/whisper_dataset
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/audio_tokenizer_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/whisper-small/res

## Step 6: Backup Download
If you did NOT connect Google Drive in Step 0, run this cell to download the model as a Zip file directly to your computer.

In [ ]:
!zip -r meetbuddy-whisper-small-finetuned.zip meetbuddy-whisper-small-finetuned/
from google.colab import files
files.download('meetbuddy-whisper-small-finetuned.zip')